# Module 4 - Topic 3: Source Citation in Answers
**AI Application Development - Developer Track (NIGCOMSAT)**

In this demo you take the retrieval pipeline from Topic 2 (metadata filtering + re-ranking) and add source citations. The LLM cites its sources, we parse and verify the citations, and we handle refusal when the corpus has no answer. One concept per cell; run them in order.

**Prerequisite:** Run the Topic 2 notebook first — this notebook reuses its ChromaDB collection.

## Setup
Connect to the ChromaDB collection created in Topic 2 and set up both the embedding client (for queries) and the LLM client (for generating answers).

In [1]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb import EmbeddingFunction, Documents, Embeddings

# Load API keys from .env in this folder (copy from Topic 2 if needed)
load_dotenv()

# --- Embedding client (same setup as Topic 2) ---
# Used by ChromaDB to embed queries before similarity search
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
embedding_client = OpenAI(
    api_key=os.getenv("EMBEDDING_API_KEY"),
    base_url=os.getenv("EMBEDDING_BASE_URL"),
)


class CorpusEmbeddingFunction(EmbeddingFunction):
    """Wraps the embedding API so ChromaDB can call it during queries."""

    def __init__(self):
        pass

    def __call__(self, input: Documents) -> Embeddings:
        response = embedding_client.embeddings.create(
            model=EMBEDDING_MODEL, input=input
        )
        return [item.embedding for item in response.data]


# --- LLM client ---
# Used to generate answers from retrieved chunks
llm_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("MODEL_NAME")

# --- Connect to Topic 2's ChromaDB collection ---
# No re-ingestion needed — we reuse the 90 chunks already stored
CHROMA_PATH = os.path.join("..", "..", "02-metadata-filtering-and-re-ranking", "demo", "chroma_db")
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(
    name="novanet_hr", embedding_function=CorpusEmbeddingFunction()
)

print(f"Connected to collection 'novanet_hr' — {collection.count()} chunks")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM model: {MODEL}")

Connected to collection 'novanet_hr' — 90 chunks
Embedding model: Qwen/Qwen3-Embedding-0.6B
LLM model: google/gemma-4-E4B-it


## Retrieval Helpers (from Topic 2)
We bring forward the `rerank` function and wrap retrieval into a single helper so we can focus on the citation layer.

In [2]:
# Boost values per document type — policies carry more authority than FAQs
TYPE_BOOST = {"policy": 0.08, "sop": 0.05, "circular": 0.06, "faq": 0.00}


def rerank(results):
    """Reorder ChromaDB results by metadata signals (from Topic 2)."""
    ranked = []
    for chunk_id, doc, meta, dist in zip(
        results["ids"][0], results["documents"][0],
        results["metadatas"][0], results["distances"][0]
    ):
        # Convert ChromaDB distance to similarity (0 = unrelated, 1 = identical)
        similarity = 1 - dist
        score = similarity

        # Boost current documents over superseded ones
        if meta["status"] == "current":
            score += 0.10

        # Boost by document type authority
        score += TYPE_BOOST.get(meta["doc_type"], 0)

        # Boost recent documents (issued in 2025 or later)
        if meta["date_issued"] >= 20250101:
            score += 0.05

        ranked.append({
            "id": chunk_id, "text": doc, "metadata": meta,
            "similarity": similarity, "reranked_score": score
        })

    # Sort highest score first
    ranked.sort(key=lambda x: x["reranked_score"], reverse=True)
    return ranked


def retrieve(query, n_results=5):
    """Retrieve, filter to current documents, and rerank."""
    results = collection.query(
        query_texts=[query], n_results=n_results,
        where={"status": "current"}  # Only return non-superseded documents
    )
    return rerank(results), results


# Quick test — check that retrieval and reranking work
test_chunks, _ = retrieve("Am I eligible to work from home?")
for i, c in enumerate(test_chunks[:3], 1):
    m = c["metadata"]
    print(f"  {i}. [{m['doc_id']}] {m['title']} (score: {c['reranked_score']:.3f})")

  1. [HR-POL-2024-003] Remote Work Policy (score: 0.705)
  2. [HR-POL-2024-003] Remote Work Policy (score: 0.696)
  3. [HR-POL-2024-003] Remote Work Policy (score: 0.690)


## Formatting Chunks for the LLM
The LLM can only cite what it can see. We add a header line with the `doc_id` and title to each chunk so the model has a concrete identifier to reference.

In [3]:
def format_chunk_for_llm(chunk):
    """Add a doc_id header so the LLM knows which document each chunk comes from."""
    meta = chunk["metadata"]
    # This header line is what the LLM will reference in its citations
    header = f"[{meta['doc_id']}] {meta['title']}"
    return f"--- {header} ---\n{chunk['text']}"


def build_context(chunks, top_n=5):
    """Join the top N formatted chunks into one string for the LLM prompt."""
    return "\n\n".join(format_chunk_for_llm(c) for c in chunks[:top_n])


# Preview: this is what the LLM receives as its source material
chunks, _ = retrieve("Am I eligible to work from home?")
context = build_context(chunks)
print(context[:600])
print("...")

--- [HR-POL-2024-003] Remote Work Policy ---
This policy defines the terms under which employees may work
remotely, the equipment and support provided, and the obligations
that apply while working outside the office.

--- [HR-POL-2024-003] Remote Work Policy ---
5. WORKING HOURS AND AVAILABILITY

--- [HR-POL-2024-003] Remote Work Policy ---
7. TERMINATION OF REMOTE WORK ARRANGEMENT

--- [HR-POL-2024-003] Remote Work Policy ---
(a) Employees who have completed their probation period (6 months)
    and whose role does not require daily physical presence may
    apply for remote work.

(b) Roles 
...


## Without Citations: The Problem
First, let us ask the LLM a question with a generic system prompt and no citation instructions. The answer is likely correct, but unsourced — you cannot verify it.

In [5]:
# Ask the same question WITHOUT citation instructions to show the problem
question = "Am I eligible to work from home?"
chunks, _ = retrieve(question)
context = build_context(chunks)

response = llm_client.chat.completions.create(
    model=MODEL,
    messages=[
        # Generic prompt — no instruction to cite sources
        {"role": "system", "content": "Answer the employee's question using the provided documents."},
        {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {question}"},
    ],
    temperature=0.1,  # Low temperature for consistent, factual answers
)

answer_no_cite = response.choices[0].message.content
print("ANSWER (no citation instructions):")
print(answer_no_cite)
print("\n--- Problem: Where did this information come from? ---")

ANSWER (no citation instructions):
To determine if you are eligible to work from home, you must meet the following criteria according to the Remote Work Policy:

*   **Probation:** You must have completed your 6-month probation period.
*   **Role Requirements:** Your role must not require a daily physical presence. Please note that the following roles are specifically excluded from remote work:
    *   Facilities staff
    *   Front-desk and reception
    *   Hardware engineers who maintain on-site equipment
    *   Security personnel
*   **Approval:** Your arrangement must be approved by both your line manager and your Head of Department.

--- Problem: Where did this information come from? ---


## The System Prompt
We instruct the LLM to cite every factual claim with a `[Source N]` marker and list the sources at the end. We also give it exact wording for when the documents do not contain an answer.

In [7]:
# The system prompt tells the LLM:
# 1. Use ONLY the provided documents (no outside knowledge)
# 2. Mark every fact with [Source N]
# 3. List all sources at the end with doc_id and section
# 4. Refuse clearly when the documents don't cover the question
# The example format helps the LLM produce consistent, parseable citations.

SYSTEM_PROMPT = """You are a NovaNet Technologies HR assistant.
Answer the employee's question using ONLY the documents below.
For each fact, add a [Source N] marker. After the answer, list sources like this:

Sources:
[Source 1] HR-POL-2025-006 — Annual Leave Policy, Section 3
[Source 2] HR-CIR-2025-001 — Revised Leave Entitlements

If the documents do not contain the answer, say:
"I don't have information about that in the HR documents I can access."
Never invent facts or cite a document not provided above."""

print(SYSTEM_PROMPT)

You are a NovaNet Technologies HR assistant.
Answer the employee's question using ONLY the documents below.
For each fact, add a [Source N] marker. After the answer, list sources like this:

Sources:
[Source 1] HR-POL-2025-006 — Annual Leave Policy, Section 3
[Source 2] HR-CIR-2025-001 — Revised Leave Entitlements

If the documents do not contain the answer, say:
"I don't have information about that in the HR documents I can access."
Never invent facts or cite a document not provided above.


## With Citations: The Improvement
Same question, same chunks — but now the system prompt requires citations and the chunks have doc_id headers. Compare the answer to the uncited version above.

In [8]:
# Same question and chunks, but now with the citation system prompt
question = "Am I eligible to work from home?"
chunks, _ = retrieve(question)
context = build_context(chunks)

response = llm_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},  # Citation-aware prompt
        {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {question}"},
    ],
    temperature=0.1,
)

answer_cited = response.choices[0].message.content
print("ANSWER (with citation instructions):")
print(answer_cited)

ANSWER (with citation instructions):
You may be eligible to apply for remote work if you have completed your 6-month probation period and your role does not require a daily physical presence [Source 1]. 

However, certain roles are excluded from remote work, including:
* Facilities staff [Source 1]
* Front-desk and reception [Source 1]
* Hardware engineers who maintain on-site equipment [Source 1]
* Security personnel [Source 1]

If you meet the eligibility criteria, your remote work arrangement must be approved by both your line manager and the Head of Department [Source 1].

Sources:
[Source 1] [HR-POL-2024-003] Remote Work Policy


## Parsing Citations
We extract the `[Source N]` references from the LLM's response into structured data — doc_id and a description — so we can verify each one.

In [9]:
# Regex pattern matching NovaNet doc IDs like HR-POL-2024-003 or HR-CIR-2025-001
DOC_ID_PATTERN = r'[A-Z]{2}-[A-Z]+-\d{4}-\d{3}'


def extract_citations(response_text):
    """Parse [Source ...] lines from the LLM response.

    LLMs produce citations in varying formats, so this parser is flexible:
      [Source 1] HR-POL-2024-003 — description
      [Source 1] [HR-POL-2024-003] description
      [Source HR-POL-2024-003] description
    We find any line starting with [Source and extract the doc_id from it.
    """
    citations = {}
    n = 0
    for line in response_text.split("\n"):
        line = line.strip()
        # Skip lines that aren't source citations
        if not re.match(r'^\[Source', line):
            continue
        # Find all doc_id patterns in this line (take the first one)
        doc_ids = re.findall(DOC_ID_PATTERN, line)
        if not doc_ids:
            continue
        n += 1
        doc_id = doc_ids[0]
        # Extract the description after the doc_id (strip punctuation)
        desc = re.split(DOC_ID_PATTERN, line)[-1]
        desc = re.sub(r'^[\s\]\[,—–―\-]+', '', desc).strip()
        citations[f"Source {n}"] = {"doc_id": doc_id, "section": desc}
    return citations


# Test the parser on the cited answer from the previous cell
citations = extract_citations(answer_cited)
print(f"Found {len(citations)} citation(s):\n")
for key, cit in citations.items():
    print(f"  {key}: {cit['doc_id']} — {cit['section']}")

Found 1 citation(s):

  Source 1: HR-POL-2024-003 — Remote Work Policy


## Verifying Citations
A citation is only trustworthy if the cited document was actually in the chunks we retrieved. If the LLM cites a doc_id that was not in its context, it hallucinated the source.

In [10]:
def verify_citations(citations, retrieved_chunks):
    """Check each citation against the chunks we actually retrieved.
    
    If the LLM cites a doc_id that wasn't in its context, it hallucinated
    the source — we flag it as UNVERIFIED.
    """
    # Build a set of doc_ids from the chunks we gave the LLM
    retrieved_doc_ids = set(c["metadata"]["doc_id"] for c in retrieved_chunks)

    verified = []
    for key, cit in citations.items():
        doc_id = cit["doc_id"]
        if doc_id in retrieved_doc_ids:
            verified.append({**cit, "key": key, "status": "VERIFIED"})
        else:
            verified.append({**cit, "key": key, "status": "UNVERIFIED",
                             "reason": "Document not in retrieved chunks"})
    return verified


# Verify the citations from the remote work answer
verified = verify_citations(citations, chunks)
print("Verification results:\n")
for v in verified:
    status_icon = "OK" if v["status"] == "VERIFIED" else "FAIL"
    print(f"  [{status_icon}] {v['key']}: {v['doc_id']} — {v['status']}")

# Demonstrate what happens with a hallucinated citation
print("\n--- Testing with a fake citation ---")
fake = {"Source 99": {"doc_id": "HR-POL-2024-007", "section": "Does not exist"}}
fake_result = verify_citations(fake, chunks)
for v in fake_result:
    print(f"  [FAIL] {v['key']}: {v['doc_id']} — {v['status']}: {v.get('reason', '')}")

Verification results:

  [OK] Source 1: HR-POL-2024-003 — VERIFIED

--- Testing with a fake citation ---
  [FAIL] Source 99: HR-POL-2024-007 — UNVERIFIED: Document not in retrieved chunks


## Refusal: When the Corpus Has No Answer
We ask a question the NovaNet HR corpus cannot answer. The LLM should refuse, and the similarity scores should be low. We test both signals.

In [11]:
# Ask something the HR corpus cannot answer — pets are not covered
refusal_question = "Can I bring my pet to the office?"
refusal_chunks, raw_results = retrieve(refusal_question)

# The similarity scores should be low since no document discusses pets
print(f"Query: \"{refusal_question}\"\n")
print("Top chunks returned:")
for i, c in enumerate(refusal_chunks[:3], 1):
    m = c["metadata"]
    print(f"  {i}. [{m['doc_id']}] {m['title']}")
    print(f"     Similarity: {c['similarity']:.3f} (low — not about pets)")

best_sim = max(c["similarity"] for c in refusal_chunks)
print(f"\nBest similarity: {best_sim:.3f}")

# Ask the LLM — the system prompt should make it refuse
context = build_context(refusal_chunks)
response = llm_client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {refusal_question}"},
    ],
    temperature=0.1,
)
refusal_answer = response.choices[0].message.content
print(f"\nLLM response:\n{refusal_answer}")

Query: "Can I bring my pet to the office?"

Top chunks returned:
  1. [HR-POL-2024-003] Remote Work Policy
     Similarity: 0.459 (low — not about pets)
  2. [HR-POL-2025-005] Travel and Expense Policy
     Similarity: 0.398 (low — not about pets)
  3. [HR-POL-2024-003] Remote Work Policy
     Similarity: 0.423 (low — not about pets)

Best similarity: 0.459

LLM response:
I don't have information about that in the HR documents I can access.


## The `should_refuse` Function
Two signals trigger a refusal: (1) the best chunk similarity is below a threshold, meaning the corpus probably does not cover this topic, and (2) the LLM produced no citations, meaning it could not find supporting evidence.

In [12]:
# Standard refusal message — friendly but clear
REFUSAL_MESSAGE = (
    "I don't have information about that in the HR documents I can access. "
    "You may want to check with the HR department directly."
)


def should_refuse(chunks, citations, threshold=0.35):
    """Decide whether the system should refuse to answer.
    
    Two independent signals (defence in depth):
    1. Low similarity — the corpus probably doesn't cover this topic
    2. No citations — the LLM couldn't find supporting evidence
    Either one alone is enough to trigger refusal.
    """
    if not chunks:
        return True, "No chunks retrieved"

    best_similarity = max(c["similarity"] for c in chunks)
    if best_similarity < threshold:
        return True, f"Best similarity {best_similarity:.3f} < threshold {threshold}"

    if not citations:
        return True, "LLM produced no citations"

    return False, "Answer supported"


# Test on the refusal question (should refuse)
refusal_cites = extract_citations(refusal_answer)
refuse, reason = should_refuse(refusal_chunks, refusal_cites)
print(f"Question: \"{refusal_question}\"")
print(f"Refuse: {refuse}  |  Reason: {reason}")

# Test on the answerable question (should NOT refuse)
print(f"\nQuestion: \"Am I eligible to work from home?\"")
refuse2, reason2 = should_refuse(chunks, citations)
print(f"Refuse: {refuse2}  |  Reason: {reason2}")

Question: "Can I bring my pet to the office?"
Refuse: True  |  Reason: LLM produced no citations

Question: "Am I eligible to work from home?"
Refuse: False  |  Reason: Answer supported


## The Full Pipeline
One function that ties everything together: retrieve → rerank → format → LLM → parse → verify → refuse or return.

In [13]:
def get_cited_answer(question, n_results=5, top_n=5, threshold=0.35):
    """Full citation pipeline: retrieve → generate → parse → verify → refuse or return."""

    # Step 1: Retrieve and rerank relevant chunks
    chunks, raw_results = retrieve(question, n_results=n_results)

    # Step 2: Format chunks with doc_id headers for the LLM
    context = build_context(chunks, top_n=top_n)

    # Step 3: Generate an answer with citation instructions
    response = llm_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {question}"},
        ],
        temperature=0.1,
    )
    answer = response.choices[0].message.content

    # Step 4: Parse citations from the response, then verify against retrieved chunks
    citations = extract_citations(answer)
    verified = verify_citations(citations, chunks[:top_n])

    # Step 5: Decide whether to refuse (low similarity or no citations)
    refuse, reason = should_refuse(chunks, citations, threshold)

    return {
        "question": question,
        "answer": REFUSAL_MESSAGE if refuse else answer,
        "refused": refuse,
        "reason": reason,
        "citations": verified,
        "best_similarity": max(c["similarity"] for c in chunks) if chunks else 0,
    }


# Quick test with an answerable question
result = get_cited_answer("Am I eligible to work from home?")
print(f"Refused: {result['refused']}")
print(f"Best similarity: {result['best_similarity']:.3f}")
print(f"Citations: {len(result['citations'])}")
print(f"\nAnswer:\n{result['answer']}")

Refused: False
Best similarity: 0.525
Citations: 1

Answer:
You may be eligible to apply for remote work if you have completed your 6-month probation period and your role does not require a daily physical presence [Source 1]. 

However, certain roles are excluded from remote work, including:
* Facilities staff [Source 1]
* Front-desk and reception [Source 1]
* Hardware engineers who maintain on-site equipment [Source 1]
* Security personnel [Source 1]

If you meet the eligibility criteria, your remote work arrangement must be approved by both your line manager and the Head of Department [Source 1].

Sources:
[Source 1] [HR-POL-2024-003] Remote Work Policy


## Running the Test Set
Five answerable questions and three refusal questions. For each, we check: did the pipeline produce a cited answer or a refusal? Were the citations verified?

In [14]:
# 5 answerable questions + 3 refusal questions
# For each, we check whether the pipeline's decision (answer or refuse) matches
TEST_SET = [
    {"q": "How many days of annual leave do I get?", "expect": "answer"},
    {"q": "Am I eligible to work from home?", "expect": "answer"},
    {"q": "What is the notice period for a Grade 6 employee?", "expect": "answer"},
    {"q": "Can I carry over unused leave?", "expect": "answer"},
    {"q": "What is the monthly remote work allowance?", "expect": "answer"},
    {"q": "Can I bring my pet to the office?", "expect": "refuse"},
    {"q": "What is the CEO's salary?", "expect": "refuse"},
    {"q": "Does NovaNet have a gym or fitness benefit?", "expect": "refuse"},
]

print(f"{'#':<3} {'Question':<48} {'Expected':<10} {'Got':<10} {'Cites':<6} {'Sim':<6} {'Result'}")
print("=" * 110)

correct = 0
for i, t in enumerate(TEST_SET, 1):
    result = get_cited_answer(t["q"])
    got = "refuse" if result["refused"] else "answer"
    match = got == t["expect"]
    if match:
        correct += 1

    # Show citation count, flagging any unverified ones with (!)
    n_cites = len(result["citations"])
    unverified = sum(1 for c in result["citations"] if c["status"] == "UNVERIFIED")
    cite_str = f"{n_cites}" if unverified == 0 else f"{n_cites} ({unverified}!)"

    q_short = (t["q"][:46] + "..") if len(t["q"]) > 48 else t["q"]
    print(f"{i:<3} {q_short:<48} {t['expect']:<10} {got:<10} {cite_str:<6} {result['best_similarity']:.3f} {'OK' if match else 'MISS'}")

print("=" * 110)
print(f"Score: {correct}/{len(TEST_SET)}")
if correct == len(TEST_SET):
    print("All questions handled correctly.")
else:
    print("Some questions need attention — check the threshold or system prompt.")

#   Question                                         Expected   Got        Cites  Sim    Result
1   How many days of annual leave do I get?          answer     answer     2      0.640 OK
2   Am I eligible to work from home?                 answer     answer     1      0.525 OK
3   What is the notice period for a Grade 6 employ.. answer     answer     1      0.728 OK
4   Can I carry over unused leave?                   answer     answer     2      0.567 OK
5   What is the monthly remote work allowance?       answer     answer     2      0.664 OK
6   Can I bring my pet to the office?                refuse     refuse     0      0.459 OK
7   What is the CEO's salary?                        refuse     refuse     0      0.476 OK
8   Does NovaNet have a gym or fitness benefit?      refuse     refuse     0      0.534 OK
Score: 8/8
All questions handled correctly.


## Recap

- **Chunk headers:** we added `[doc_id] title` headers so the LLM has identifiers to cite
- **System prompt:** instructs the LLM to cite every claim with `[Source N]` and refuse when no source exists
- **Parsing:** regex extracts doc_id and section from the response
- **Verification:** checks that each cited document was in the retrieved chunks — catches hallucinated citations
- **Refusal:** low similarity + no citations = refuse to answer
- **Full pipeline:** `get_cited_answer()` ties it all together

This becomes `citations.py` in the reference application. Next topic — **Conversation History & Context Management** — handles follow-up questions: when the employee says "Does that apply to contract staff?" the system rewrites it into a standalone query before retrieval.